# LoRA / QLoRA Concept Demo Notebook

Hands-on confirmation of the theory from `lora_qlora.md` using **numpy only** — no GPU required, runnable as-is.

Topics covered:
1. Low-rank decomposition ΔW = B·A and reconstruction error
2. Parameter count reduction in LoRA (vary r and observe)
3. Memory breakdown comparison: Full / LoRA / QLoRA
4. NF4 quantization simulation (error comparison with int4)
5. Double Quantization overhead calculation

> For real training on hardware, see `lora_qlora_finetune.ipynb` (Colab/GPU).

In [ ]:
# Install dependencies if not already installed
# %pip install numpy matplotlib
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print('ready')

---

## 1. Low-Rank Decomposition ΔW = B·A

If the fine-tuning weight update ΔW is low-rank, it can be approximated as `B(d×r)·A(r×k)`.
Truncate the rank with SVD and examine reconstruction error.

In [ ]:
d, k = 256, 256
# Artificially create a ΔW with low intrinsic rank (true rank=8)
true_rank = 8
B_true = np.random.randn(d, true_rank)
A_true = np.random.randn(true_rank, k)
dW = B_true @ A_true + 0.01*np.random.randn(d, k)  # slight noise

def low_rank_approx(M, r):
    U, S, Vt = np.linalg.svd(M, full_matrices=False)
    return (U[:, :r] * S[:r]) @ Vt[:r, :]

def rel_error(M, Mr):
    return np.linalg.norm(M - Mr) / np.linalg.norm(M)

ranks = [1, 2, 4, 8, 16, 32, 64, 128]
errs = [rel_error(dW, low_rank_approx(dW, r)) for r in ranks]
for r, e in zip(ranks, errs):
    print(f'r={r:3d}  relative_error={e:.4f}  param_count r*(d+k)={r*(d+k):,}')

In [ ]:
plt.figure(figsize=(7,4))
plt.plot(ranks, errs, 'o-')
plt.axvline(true_rank, color='r', ls='--', label=f'True rank={true_rank}')
plt.xlabel('LoRA rank r'); plt.ylabel('Relative reconstruction error')
plt.title('Low-rank approximation: error drops sharply when r reaches the true rank')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

---

## 2. Parameter Count Reduction in LoRA

Compare the number of trainable parameters between Full and LoRA for a single Linear layer with `d=k=4096` (7B-class).

In [ ]:
def full_params(d, k):  return d*k
def lora_params(d, k, r): return r*(d+k)

d = k = 4096
print(f'Full:           {full_params(d,k):,}')
for r in [4, 8, 16, 32, 64]:
    lp = lora_params(d, k, r)
    pct = 100*lp/full_params(d,k)
    print(f'LoRA (r={r:2d}):   {lp:>10,}   ({pct:.3f}% of full)')

In [ ]:
# Initialization insight: B=0 means ΔW=0 at training start -> identical output to the base model
r = 8
A = np.random.randn(r, k) * (1/np.sqrt(k))   # kaiming-style
Bz = np.zeros((d, r))                          # B=0 initialization
dW0 = Bz @ A
print('||ΔW|| with B=0 initialization =', np.linalg.norm(dW0), '-> perfectly matches base model at training start')

---

## 3. Memory Breakdown: Full vs LoRA vs QLoRA

Calculate the breakdown of weights, gradients, and Adam state for a 7B model and plot as a bar chart (activations excluded).

In [ ]:
N = 7e9              # 7B params
trainable_ratio = 0.004   # LoRA: ~0.4%

def breakdown(mode):
    if mode == 'Full FT (bf16)':
        w = 2*N                     # bf16 weights
        g = 2*N                     # gradients (all params)
        opt = 8*N                   # Adam m,v in fp32 (4+4)
    elif mode == 'LoRA (bf16)':
        w = 2*N                     # weights frozen but kept
        g = 2*N*trainable_ratio
        opt = 8*N*trainable_ratio
    elif mode == 'QLoRA (NF4)':
        w = 0.5*N + 0.127/8*N       # 4-bit weights + DoubleQuant constants
        g = 2*N*trainable_ratio
        opt = 8*N*trainable_ratio
    return np.array([w, g, opt]) / 1e9   # GB

modes = ['Full FT (bf16)', 'LoRA (bf16)', 'QLoRA (NF4)']
data = {m: breakdown(m) for m in modes}
for m in modes:
    w,g,o = data[m]
    print(f'{m:16s} weights={w:6.2f}GB gradients={g:6.3f}GB Adam={o:6.3f}GB total={w+g+o:6.2f}GB')

In [ ]:
labels = ['Weights', 'Gradients', 'Adam state']
x = np.arange(len(modes)); bottom = np.zeros(len(modes))
plt.figure(figsize=(8,5))
for i, lab in enumerate(labels):
    vals = np.array([data[m][i] for m in modes])
    plt.bar(x, vals, bottom=bottom, label=lab)
    bottom += vals
plt.xticks(x, modes); plt.ylabel('GPU Memory (GB)')
plt.title('Memory Breakdown for 7B Fine-tuning (activations excluded)')
plt.legend(); plt.grid(True, axis='y', alpha=0.3)
for i, m in enumerate(modes):
    plt.text(i, bottom[i]+1, f'{bottom[i]:.1f}GB', ha='center')
plt.show()

---

## 4. NF4 Quantization Simulation

Neural network weights are close to a zero-mean normal distribution. NF4 uses **non-uniform 4-bit grid points aligned with normal distribution quantiles**.
Compare reconstruction error against int4 (uniform spacing) with the same 4 bits.

In [ ]:
# NF4 16 grid points (from bitsandbytes create_normal_map)
NF4 = np.array([
    -1.0, -0.6961928, -0.5250731, -0.3949175, -0.2844414, -0.1847734,
    -0.0910500,  0.0,  0.0795803,  0.1609302,  0.2461123,  0.3379152,
     0.4407098,  0.5626170,  0.7229568,  1.0])
INT4 = np.linspace(-1, 1, 16)   # uniform spacing (same 16 values = 4-bit)

def quantize_blockwise(x, levels, block=64):
    x = x.flatten().astype(np.float64)
    out = np.empty_like(x)
    for s in range(0, len(x), block):
        blk = x[s:s+block]
        absmax = np.abs(blk).max() + 1e-12
        norm = blk / absmax                       # normalize to [-1, 1]
        idx = np.abs(norm[:,None] - levels[None,:]).argmin(1)  # nearest grid point
        out[s:s+block] = levels[idx] * absmax     # dequantize
    return out

W = np.random.randn(4096*64)   # normally distributed weights
W_nf4  = quantize_blockwise(W, NF4)
W_int4 = quantize_blockwise(W, INT4)
mse = lambda a,b: np.mean((a-b)**2)
print(f'NF4  reconstruction MSE = {mse(W, W_nf4):.6e}')
print(f'int4 reconstruction MSE = {mse(W, W_int4):.6e}')
print(f'NF4 error is {mse(W,W_int4)/mse(W,W_nf4):.2f}x smaller than int4')

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(W, bins=120, density=True, alpha=0.4, label='Weight distribution N(0,1)')
for v in NF4:  plt.axvline(v, color='g', alpha=0.5, lw=0.8)
for v in INT4: plt.axvline(v, color='r', alpha=0.5, lw=0.8, ls='--')
plt.plot([],[], color='g', label='NF4 grid (quantile-based)')
plt.plot([],[], color='r', ls='--', label='int4 grid (uniform)')
plt.xlim(-3,3); plt.legend(); plt.title('NF4 concentrates grid points at the high-density center')
plt.xlabel('Normalized weight value'); plt.show()

---

## 5. Double Quantization Overhead

Block-wise quantization requires a quantization constant (scale) per block. Calculate the bits/param cost.

In [ ]:
def scale_overhead_bits(block, scale_bytes):
    return scale_bytes*8 / block   # divide scale cost (bytes) over block size

block = 64
naive = scale_overhead_bits(block, 4)            # fp32 scale
# Double Quant: quantize scales to 8-bit + upper-level scale (fp32 per 256 blocks)
dq = scale_overhead_bits(block, 1) + scale_overhead_bits(block*256, 4)
print(f'Naive (fp32 scale):      {naive:.3f} bit/param')
print(f'Double Quantization:     {dq:.3f} bit/param')
print(f'Savings:                 {naive-dq:.3f} bit/param')
print(f'For 7B: {(naive-dq)*7e9/8/1e9:.2f} GB additional reduction')

---

## 6. Summary: Key Numbers

What this notebook demonstrated:
- ΔW error drops to near zero once the true rank is reached → **low-rank approximation is justified**
- With r=8, trainable parameters reduce to **0.39%**
- QLoRA compresses a 7B model to **~3.8 GB** (Full 84 GB → less than 1/20)
- NF4 has smaller reconstruction error than int4 (because weights follow a normal distribution)

Next, run actual fine-tuning in `lora_qlora_finetune.ipynb`.